In [1]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## input

In [4]:
from src.preprocessing import pp
from sklearn.model_selection import train_test_split
import scvi

In [5]:
control_key = "is_control"
condition_rep_keys = "perturbation_embeddings"
condition_combined_keys = "condition_combined"
mass_deduct_keys = "plate_well" # or None
random_seed = 42

condition_keys = "perturbation" # 数据集perturbation所对应的obs列名
dataset_name = "Sciplex3_chembert"
sample_rep = "X_pca" #"X_scVI"  "X_flatvi" "X_state"

cov_config = {
    "cell_line": {
        "type": "categorical",
        "control_ot": "groupwise",
        "perturbed_ot": "groupwise",
        "use_in_model": True,
        "model_source": "both",
        "contain_in_condition": True,
        "condition_source": "both",
    },
    # "celltype":{
    #     "type": "categorical",
    #     "control_ot": "global",
    #     "perturbed_ot": "global",
    #     "use_in_model": True,
    #     "model_source": "control",
    #     "contain_in_condition":False,
    #     "condition_source": None,
    # },
    "dose_value": {
        "type": "continuous",
        "transform": "log1p_zscore",
        "control_ot": "global",
        "perturbed_ot": "groupwise",
        "use_in_model": True,
        "model_source": "perturbed",
        "contain_in_condition": True,
        "condition_source": "perturbed",
    },
    "time": {
        "type": "continuous",
        "transform": "log1p_zscore",   
        "control_ot": "groupwise",
        "perturbed_ot": "groupwise",
        "use_in_model": True,
        "model_source": "both",
        "contain_in_condition": True,
        "condition_source": "both",
    },
}


if_adata_ref = None #用于根据一个参考adata快速构建pca
adata_ref_path = "data/processed/PBMC2000_pca_rep_0.2_42.h5ad"

condition_rep_dict = pd.read_pickle("./data/processed/drug_embeddings_sciplex3_chembert.pkl")
condition_rep_dict = condition_rep_dict["drug_to_embedding"]

In [6]:
filePath = './data/raw/Sciplex3_hvg2000.h5ad'
adata = sc.read_h5ad(filePath)
adata.uns["cov_config"] = cov_config
print(adata)

AnnData object with n_obs × n_vars = 762795 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type'
    var: 'ensembl_id', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'cov_config'
    layers: 'counts'


In [7]:
adata.obs[control_key] = (adata.obs[condition_keys] == "control")
condition_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

is_control
False    745217
True      17578
Name: count, dtype: int64


In [8]:
adata.obs[mass_deduct_keys] = adata.obs["plate"].astype(str) + "_" + adata.obs["well"].astype(str) #处理mass

## splitting

In [9]:
rng = np.random.default_rng(random_seed) 
test_ratio = 0.2
condition_list = list(condition_list)
zero_shot = True

if not zero_shot:
    # 分层抽样 先验证分布内学习能力
    pert_mask = adata.obs[control_key] == False
    y = adata.obs.loc[pert_mask, condition_keys].astype(str).values
    pert_indices = np.flatnonzero(pert_mask)

    train_idx, test_idx = train_test_split(
        pert_indices,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y 
    )
    adata_train = adata[train_idx].copy()
    adata_test = adata[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()
    adata_train.uns["normalized_m"] = 1 / (1-test_ratio)
    adata_test.uns["normalized_m"] = 1 / test_ratio
    adata_control.uns["normalized_m"] = 1 
    print(condition_list)
else:
    # 按condition分割 zeroshot
    n_test = max(1, int(len(condition_list) * test_ratio))
    test_condition = rng.choice(condition_list, size=n_test, replace=False).tolist()
    print(test_condition)
    train_condition = [g for g in condition_list if g not in test_condition]
    print(train_condition)
    adata_control = adata[adata.obs[control_key]==True].copy() # control的target_condition是 PBS
    adata_train = adata[adata.obs[condition_keys].isin(train_condition)].copy() 
    adata_test = adata[adata.obs[condition_keys].isin(test_condition)].copy()
    adata_train.uns["normalized_m"] = 1
    adata_test.uns["normalized_m"] = 1
    adata_control.uns["normalized_m"] = 1 

['Filgotinib (GLPG0634)', 'Baricitinib (LY3009104, INCB028050)', 'Mesna', 'EED226', 'Vandetanib (ZD6474)', 'INO-1001 (3-Aminobenzamide)', 'Trichostatin A (TSA)', 'CYC116', 'Ofloxacin', 'Capecitabine', 'Veliparib (ABT-888)', 'PF-573228', 'Divalproex Sodium', 'Cimetidine', 'Rucaparib (AG-014699,PF-01367338) phosphate', 'Pelitinib (EKB-569)', 'SRT2104 (GSK2245840)', 'Dasatinib', 'SRT3025 HCl', 'GSK1070916', 'BMS-536924', 'Linifanib (ABT-869)', 'Navitoclax (ABT-263)', 'Costunolide', 'Temsirolimus (CCI-779, NSC 683864)', 'Pirarubicin', 'Decitabine', 'Danusertib (PHA-739358)', 'BMS-754807', 'Quercetin', 'Ruxolitinib (INCB018424)', 'Tazemetostat (EPZ-6438)', 'Ki8751', 'Tranylcypromine (2-PCPA) HCl', 'Celecoxib', 'Lenalidomide (CC-5013)', 'ZM 447439']
['TAK-901', 'AG-490 (Tyrphostin B42)', 'Abexinostat (PCI-24781)', 'Alisertib (MLN8237)', 'Busulfan', 'Obatoclax Mesylate (GX15-070)', 'Enzastaurin (LY317615)', 'BMS-265246', 'UNC0379', 'Raltitrexed', 'Tubastatin A HCl', 'AG-14361', 'Iniparib (BSI

In [10]:
del adata

## latent embedding

In [11]:
n_comps = 100
n_hidden = 1024
n_layers = 2
#condition_rep_dict = pd.read_pickle("./data/processed/PBMC_cytokines.pkl")
model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}"
flatvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}"
state_decoder_save_path = f"./data/processed/model/{sample_rep}_{dataset_name}"
model_save_path = None

load_embedding_model = True
if sample_rep in ["X_scVI"]:
    model_save_path = scvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{model_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{model_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{model_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep == "X_flatvi":
    model_save_path = flatvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{flatvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
elif sample_rep=="X_state":
    from src.preprocessing import build_train_eval_loaders,NBDecoderTrainer,NBDecoder
    model_save_path = state_decoder_save_path
    z_dim = adata_control.obsm["X_state"].shape[1] # 2058
    n_genes = adata_control.n_vars
    decoder = NBDecoder(z_dim=z_dim, n_genes=n_genes, hidden=(1024,2048,4096), dropout=0.1)
    train_loader, val_loader = build_train_eval_loaders(
                                    adata_train=adata_control,
                                    adata_eval=adata_train,   
                                    count_layer="counts",
                                    emb_key="X_state",
                                    batch_size=256,
                                )
    trainer = NBDecoderTrainer(decoder, lr=1e-4, device="cuda", use_amp=True)
    trainer.fit(train_loader, val_loader=val_loader, epochs=50)
    trainer.save(f"{state_decoder_save_path}.pt")

In [12]:
if if_adata_ref:
    adata_ref = sc.read_h5ad(adata_ref_path,backed='r')
else:
    adata_ref = None

In [13]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_ref = adata_ref,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    n_layers = n_layers,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    model_save_path = model_save_path,
    control_key = control_key,
    condition_keys = condition_keys,
    condition_rep_keys = condition_rep_keys,
    condition_combined_keys = condition_combined_keys,
    cov_config = cov_config,
    condition_rep_dict = condition_rep_dict,
    pca_method = "scanpy", # "parse"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )

[6.4562697  5.0786595  2.0056028  1.5977103  1.4587027  1.4402717
 1.1456429  1.2058095  1.2024679  1.1701125  1.1283458  1.127593
 1.100503   1.0739735  1.0576932  1.010951   1.0253536  1.0098143
 1.0080787  0.9695171  0.9621695  0.9440671  0.92211235 0.93770957
 0.90365833 0.924672   0.9121938  0.9125819  0.8985118  0.91117793
 0.92194736 0.89418495 0.8953304  0.8802131  0.9008848  0.901762
 0.88278085 0.8778881  0.8775404  0.8777958  0.871185   0.8691746
 0.8585499  0.85899246 0.86762863 0.86229175 0.844441   0.8499642
 0.84332764 0.8383991  0.83912146 0.833336   0.82613575 0.8278163
 0.8286331  0.8220861  0.82055193 0.811475   0.8176162  0.81460685
 0.8062436  0.798779   0.7957245  0.8032861  0.7932558  0.7960841
 0.779449   0.78161883 0.79742    0.79152244 0.79372525 0.7902077
 0.7668443  0.77344507 0.7890173  0.7816119  0.77855945 0.7666339
 0.76909953 0.7734347  0.7620233  0.76012224 0.75254023 0.75832146
 0.75169575 0.7491127  0.73998237 0.755638   0.74326515 0.7453973
 0.73459

In [14]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep}_{n_comps}_{if_adata_ref}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [15]:
print(preprocess_save_path)
print(adata_control)
print(adata_train)
print(adata_test)

./data/processed/Sciplex3_chembert_42_0.2_True_X_pca_100_None
AnnData object with n_obs × n_vars = 17578 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type', 'is_control', 'plate_well', 'cell_line_idx', 'dose_value_scaled', 'time_scaled', 'condition_combined'
    var: 'ensembl_id', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'cov_config', 'normalized_m', 'pca', 'global_rulebook'
    obsm: 'X_pca'
    varm: 'PCs', 'X_mean'
    layers: 'counts'
AnnData object with n_obs × n_vars = 591464 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type

In [64]:
adata_control.uns

{'hvg': {'flavor': 'seurat'},
 'log1p': {},
 'cov_config': {},
 'normalized_m': 0.16666666666666666,
 'pca': {'params': {'zero_center': False,
   'use_highly_variable': False,
   'mask_var': None,
   'layer': 'X_centered'},
  'variance': array([39.700928  , 25.616402  ,  4.1866107 ,  2.566561  ,  2.134013  ,
          2.0720105 ,  1.8534021 ,  1.6658701 ,  1.5192015 ,  1.356947  ,
          1.2971735 ,  1.2443907 ,  1.2161279 ,  1.1695455 ,  1.115291  ,
          1.0400641 ,  1.0295781 ,  0.9959945 ,  0.97372144,  0.93906754,
          0.92801684,  0.9014805 ,  0.883783  ,  0.8651181 ,  0.8520536 ,
          0.8423354 ,  0.8340875 ,  0.8312146 ,  0.8173032 ,  0.81567   ,
          0.807885  ,  0.80027044,  0.7892925 ,  0.78684205,  0.7806047 ,
          0.77215123,  0.75936   ,  0.7536816 ,  0.75186807,  0.74248356,
          0.7412296 ,  0.7330665 ,  0.7280949 ,  0.7216992 ,  0.719462  ,
          0.71305984,  0.7065062 ,  0.70113236,  0.69250125,  0.69051886,
          0.6820178 ,  0

In [ ]:
adata_train.obs[condition_combined_keys].value_counts()

In [ ]:
denoised_df = model_ref.get_normalized_expression(adata_control, return_mean=True,library_size=1e4)
raw_adata = adata_control.copy()
sc.pp.normalize_total(raw_adata, target_sum=1e4)
raw_matrix = raw_adata.X
if hasattr(raw_matrix, "toarray"):
    raw_matrix = raw_matrix.toarray()

In [ ]:
# 计算原始数据和重建数据的基因均值
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
mean_raw = np.mean(raw_matrix, axis=0)
mean_recon = np.mean(denoised_df.values, axis=0)

# 计算相关性 (Pearson 或 Spearman)
corr, _ = pearsonr(mean_raw, mean_recon)
print(f"Gene Mean Correlation (Raw vs Recon): {corr:.4f}")

# 可视化
plt.figure(figsize=(6, 6))
plt.scatter(mean_raw, mean_recon, s=1, alpha=0.5)
plt.plot([0, max(mean_raw)], [0, max(mean_raw)], 'r--') # 对角线
plt.xlabel("Raw Mean Expression (Normalized)")
plt.ylabel("Reconstructed Mean Expression")
plt.title(f"Reconstruction Quality (R = {corr:.2f})")
plt.xscale('log')
plt.yscale('log')
plt.show()